# YouTube Channel Data Analysis

### Data Analysis Essentials Project

This notebook performs simple data cleaning, descriptive analysis, visualization, growth prediction, and clustering.

In [ ]:
# ============================================================
# YOUTUBE CHANNEL DATA ANALYSIS
# Data Analysis Essentials Project
# ============================================================

# Install required libraries
!pip -q install pandas numpy matplotlib seaborn scikit-learn

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# 1. LOAD THE DATA
# ------------------------------------------------------------

# We do NOT assume any particular CSV file name.
# If one CSV is already in Colab, it will be used.
# If there are multiple CSV files, choose one.

from glob import glob

csv_files = [file for file in glob("*.csv") if not Path(file).name.startswith(".")]

if len(csv_files) == 0:
    from google.colab import files

    uploaded_files = files.upload()

    csv_files = [
        file for file in uploaded_files
        if file.lower().endswith(".csv")
    ]

if len(csv_files) == 0:
    raise FileNotFoundError("Please upload a CSV dataset.")

if len(csv_files) == 1:
    file_name = csv_files[0]
else:
    print("CSV files found:")
    for number, file in enumerate(csv_files, start=1):
        print(number, "-", file)

    choice = int(input("Enter the number of the CSV file: "))
    file_name = csv_files[choice - 1]

df = pd.read_csv(file_name)

print("Dataset loaded:", file_name)
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

# ------------------------------------------------------------
# 2. CLEAN COLUMN NAMES
# ------------------------------------------------------------

def clean_column_name(column):
    column = column.strip().lower()
    column = re.sub(r"[^a-z0-9_]+", "_", column)
    column = re.sub(r"_+", "_", column)
    return column.strip("_")

df.columns = [clean_column_name(column) for column in df.columns]

print("\nColumn names:")
print(df.columns.tolist())

# ------------------------------------------------------------
# 3. CLEAN THE DATA
# ------------------------------------------------------------

# These are the main numeric columns used in our analysis.
number_columns = [
    "subscribers",
    "total_views",
    "total_videos"
]

# Convert them to numbers.
for column in number_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Remove duplicate rows.
df = df.drop_duplicates()

# Remove rows where important values are missing.
df = df.dropna(subset=number_columns)

# Keep only positive values.
df = df[
    (df["subscribers"] > 0) &
    (df["total_views"] > 0) &
    (df["total_videos"] > 0)
]

print("\nData after cleaning:")
print("Rows:", len(df))

# ------------------------------------------------------------
# 4. CREATE NEW FEATURES
# ------------------------------------------------------------

# Average views received by each video.
df["views_per_video"] = (
    df["total_views"] / df["total_videos"]
)

# Average subscribers gained per video.
df["subs_per_video"] = (
    df["subscribers"] / df["total_videos"]
)

# Simple engagement indicator.
df["engagement_proxy"] = (
    df["total_views"] / df["subscribers"]
)

print("\nNew columns created:")
print("views_per_video")
print("subs_per_video")
print("engagement_proxy")

# ------------------------------------------------------------
# 5. BASIC DATA ANALYSIS
# ------------------------------------------------------------

print("\n========== BASIC STATISTICS ==========")

print(df[
    [
        "subscribers",
        "total_views",
        "total_videos",
        "views_per_video",
        "subs_per_video",
        "engagement_proxy"
    ]
].describe().round(2))

# ------------------------------------------------------------
# 6. GRAPH 1 - NUMBER OF CHANNELS
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.hist(df["subscribers"], bins=30)

plt.title("Distribution of Subscribers")
plt.xlabel("Subscribers")
plt.ylabel("Number of Channels")

plt.show()

# ------------------------------------------------------------
# 7. GRAPH 2 - TOTAL VIEWS
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.hist(df["total_views"], bins=30)

plt.title("Distribution of Total Views")
plt.xlabel("Total Views")
plt.ylabel("Number of Channels")

plt.show()

# ------------------------------------------------------------
# 8. GRAPH 3 - OUTLIER ANALYSIS
# ------------------------------------------------------------

# Log transformation makes very large YouTube values easier to compare.
box_data = np.log1p(
    df[
        [
            "subscribers",
            "total_views",
            "total_videos"
        ]
    ]
)

plt.figure(figsize=(8, 5))

sns.boxplot(data=box_data)

plt.title("Outlier Analysis")
plt.xlabel("Variables")
plt.ylabel("Log Scaled Value")

plt.show()

# ------------------------------------------------------------
# 9. GRAPH 4 - SUBSCRIBERS VS TOTAL VIEWS
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=df,
    x="subscribers",
    y="total_views"
)

plt.xscale("log")
plt.yscale("log")

plt.title("Subscribers vs Total Views")
plt.xlabel("Subscribers")
plt.ylabel("Total Views")

plt.show()

# ------------------------------------------------------------
# 10. GRAPH 5 - VIDEOS VS SUBSCRIBERS
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=df,
    x="total_videos",
    y="subscribers"
)

plt.xscale("log")
plt.yscale("log")

plt.title("Total Videos vs Subscribers")
plt.xlabel("Total Videos")
plt.ylabel("Subscribers")

plt.show()

# ------------------------------------------------------------
# 11. GRAPH 6 - VIEWS PER VIDEO
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.hist(df["views_per_video"], bins=30)

plt.title("Distribution of Views Per Video")
plt.xlabel("Views Per Video")
plt.ylabel("Number of Channels")

plt.show()

# ------------------------------------------------------------
# 12. GRAPH 7 - TOP 10 CHANNELS BY SUBSCRIBERS
# ------------------------------------------------------------

top_subscribers = df.nlargest(10, "subscribers")
top_subscribers = top_subscribers.sort_values("subscribers")

plt.figure(figsize=(10, 6))

plt.barh(
    top_subscribers["channel_name"],
    top_subscribers["subscribers"]
)

plt.title("Top 10 Channels by Subscribers")
plt.xlabel("Subscribers")
plt.ylabel("Channel")

plt.show()

# ------------------------------------------------------------
# 13. GRAPH 8 - TOP 10 CHANNELS BY TOTAL VIEWS
# ------------------------------------------------------------

top_views = df.nlargest(10, "total_views")
top_views = top_views.sort_values("total_views")

plt.figure(figsize=(10, 6))

plt.barh(
    top_views["channel_name"],
    top_views["total_views"]
)

plt.title("Top 10 Channels by Total Views")
plt.xlabel("Total Views")
plt.ylabel("Channel")

plt.show()

# ------------------------------------------------------------
# 14. GRAPH 9 - TOP 10 CHANNELS BY VIEWS PER VIDEO
# ------------------------------------------------------------

top_views_per_video = df.nlargest(
    10,
    "views_per_video"
)

top_views_per_video = top_views_per_video.sort_values(
    "views_per_video"
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_views_per_video["channel_name"],
    top_views_per_video["views_per_video"]
)

plt.title("Top 10 Channels by Views Per Video")
plt.xlabel("Views Per Video")
plt.ylabel("Channel")

plt.show()

# ------------------------------------------------------------
# 15. GRAPH 10 - COUNTRY-WISE CHANNEL COUNT
# ------------------------------------------------------------

if "country" in df.columns:

    country_count = df["country"].fillna("Unknown").value_counts().head(10)

    plt.figure(figsize=(10, 6))

    plt.bar(
        country_count.index,
        country_count.values
    )

    plt.title("Top 10 Countries by Number of Channels")
    plt.xlabel("Country")
    plt.ylabel("Number of Channels")

    plt.xticks(rotation=45)

    plt.show()

# ------------------------------------------------------------
# 16. GRAPH 11 - TIER-WISE CHANNEL COUNT
# ------------------------------------------------------------

if "tier" in df.columns:

    tier_count = df["tier"].fillna("Unknown").value_counts()

    plt.figure(figsize=(8, 5))

    plt.bar(
        tier_count.index,
        tier_count.values
    )

    plt.title("Channel Distribution by Tier")
    plt.xlabel("Tier")
    plt.ylabel("Number of Channels")

    plt.xticks(rotation=30)

    plt.show()

# ------------------------------------------------------------
# 17. GRAPH 12 - CORRELATION HEATMAP
# ------------------------------------------------------------

correlation = df[
    [
        "subscribers",
        "total_views",
        "total_videos",
        "views_per_video",
        "subs_per_video",
        "engagement_proxy"
    ]
].corr()

plt.figure(figsize=(9, 7))

sns.heatmap(
    correlation,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)

plt.title("Correlation Between Important Variables")

plt.show()

# ------------------------------------------------------------
# 18. GROUP CHANNELS FOR ML ANALYSIS
# ------------------------------------------------------------

# The dataset does not have a category column.
# Therefore, each channel is used as one analysis group.

channel_data = (
    df.groupby("channel_name")
    .agg(
        total_channels=("subscribers", "count"),
        total_subscribers=("subscribers", "sum"),
        total_views=("total_views", "sum"),
        average_subscribers=("subscribers", "mean"),
        average_views=("total_views", "mean"),
        average_views_per_video=("views_per_video", "mean"),
        average_engagement=("engagement_proxy", "mean")
    )
    .reset_index()
)

print("\nChannel analysis data:")
print(channel_data.head())

# ------------------------------------------------------------
# 19. RANDOM FOREST - GROWTH PREDICTION
# ------------------------------------------------------------

features = [
    "total_channels",
    "total_views",
    "average_views_per_video",
    "average_engagement"
]

X = channel_data[features]
y = channel_data["average_subscribers"]

# Scale the values.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Create and train the model.
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_scaled, y)

# Predict the growth score.
channel_data["growth_score"] = model.predict(X_scaled)

# Rank the channels.
channel_data["growth_rank"] = (
    channel_data["growth_score"]
    .rank(ascending=False, method="min")
)

# ------------------------------------------------------------
# 20. GRAPH 13 - GROWTH SCORE
# ------------------------------------------------------------

top_growth = channel_data.nlargest(10, "growth_score")
top_growth = top_growth.sort_values("growth_score")

plt.figure(figsize=(10, 6))

plt.barh(
    top_growth["channel_name"],
    top_growth["growth_score"]
)

plt.title("Top 10 Channels by Predicted Growth Score")
plt.xlabel("Growth Score")
plt.ylabel("Channel")

plt.show()

# ------------------------------------------------------------
# 21. K-MEANS CLUSTERING
# ------------------------------------------------------------

cluster_columns = [
    "average_views",
    "average_views_per_video",
    "average_engagement",
    "total_channels"
]

cluster_data = channel_data[cluster_columns]

cluster_scaled = StandardScaler().fit_transform(
    cluster_data
)

# Divide channels into 3 groups.
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

channel_data["cluster"] = kmeans.fit_predict(
    cluster_scaled
)

# ------------------------------------------------------------
# 22. GRAPH 14 - CHANNEL CLUSTERS
# ------------------------------------------------------------

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=channel_data,
    x="average_views_per_video",
    y="average_engagement",
    hue="cluster",
    s=100
)

plt.title("YouTube Channel Clusters")
plt.xlabel("Average Views Per Video")
plt.ylabel("Average Engagement")

plt.show()

# ------------------------------------------------------------
# 23. TRENDING INDEX
# ------------------------------------------------------------

# Standardize three important measures.
views_score = StandardScaler().fit_transform(
    channel_data[["average_views"]]
).flatten()

views_per_video_score = StandardScaler().fit_transform(
    channel_data[["average_views_per_video"]]
).flatten()

engagement_score = StandardScaler().fit_transform(
    channel_data[["average_engagement"]]
).flatten()

# Give more importance to views per video and engagement.
channel_data["trending_index"] = (
    views_per_video_score * 0.4
    + engagement_score * 0.4
    + views_score * 0.2
)

channel_data["trending_rank"] = (
    channel_data["trending_index"]
    .rank(ascending=False, method="min")
)

# ------------------------------------------------------------
# 24. GRAPH 15 - TRENDING CHANNELS
# ------------------------------------------------------------

top_trending = channel_data.nlargest(
    10,
    "trending_index"
)

top_trending = top_trending.sort_values(
    "trending_index"
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_trending["channel_name"],
    top_trending["trending_index"]
)

plt.title("Top 10 Trending Channels")
plt.xlabel("Trending Index")
plt.ylabel("Channel")

plt.show()

# ------------------------------------------------------------
# 25. FINAL RESULTS
# ------------------------------------------------------------

print("\n========== TOP CHANNELS BY SUBSCRIBERS ==========")

print(
    df.nlargest(
        10,
        "subscribers"
    )[
        [
            "channel_name",
            "subscribers",
            "total_views",
            "total_videos"
        ]
    ].to_string(index=False)
)

print("\n========== TOP CHANNELS BY GROWTH SCORE ==========")

print(
    channel_data.nlargest(
        10,
        "growth_score"
    )[
        [
            "channel_name",
            "growth_score",
            "growth_rank"
        ]
    ].to_string(index=False)
)

print("\n========== TOP TRENDING CHANNELS ==========")

print(
    channel_data.nlargest(
        10,
        "trending_index"
    )[
        [
            "channel_name",
            "trending_index",
            "trending_rank"
        ]
    ].to_string(index=False)
)

print("\n========== PROJECT COMPLETED ==========")
print("Data cleaning completed.")
print("Feature engineering completed.")
print("Data analysis graphs completed.")
print("Growth prediction completed.")
print("K-Means clustering completed.")
print("Trending analysis completed.")
